As part of my internship project, I will be maintaining a glossary of concepts I've learned during the past few weeks. 

## Feature Selection

After preprocessing, the dataset contains **724 predictor variables**, including clinical features, gene expression measurements, and mutation indicators.

Although modern ML algorithms can train on hundreds of variables, using every available feature is not always beneficial. Many variables may contain redundant information, contribute very little to prediction, or introduce noise that reduces a model's ability to generalize to unseen patients.

Feature selection aims to identify the most informative predictors while reducing dimensionality, improving interpretability, decreasing computational cost, and minimizing overfitting.

### A. Dimensionality

Dimensionality refers to the number of input variables (features) used by a machine learning model.

For example:

- 10 features -> 10-dimensional dataset
- 724 features -> 724-dimensional dataset

While adding features can increase the amount of available information, excessively high dimensionality often causes models to memorize training data instead of learning general patterns. This phenomenon is commonly known as the **curse of dimensionality** (primary cause of overfitting).

### B. Overfitting

Overfitting occurs when a machine learning model learns patterns that are specific to the training dataset rather than the underlying biological relationships.

Instead of discovering general rules that apply to new patients, an overfit model effectively memorizes noise present in the training data. Such models usually achieve excellent training performance but perform poorly on unseen data.

Reducing unnecessary features is one of the most effective ways to decrease the risk of overfitting.

### C. Variance Threshold

Variance measures how much a feature changes across different patients. Features with extremely low variance contain approx. the same value for every patient and therefore provide very little information to help distinguish between different survival outcomes.

Variance Threshold is a simple feature selection technique that removes variables whose variance falls below a chosen threshold. Example:

Gene A:
0 0 0 0 0 0 0 0

Gene B:
0 1 0 1 0 1 0 1

Gene B contains more useful information because it varies across patients.

### D. Correlation

Correlation measures the strength of the relationship between two variables. If two features are highly correlated, they often contain nearly identical information. For example: tumour size & the Nottingham Prognostic Index. These variables are biologically related. Including both may introduce redundancy without improving prediction. **Correlation filtering** removes one feature from highly correlated pairs, reducing redundancy while preserving most of the available information.

### E. Univariate Analysis

A univariate analysis evaluates one feature at a time. Instead of examining how multiple variables interact, each predictor is independently tested for its association with patient survival. Features demonstrating stronger statistical associations are more likely to be useful predictors and can be prioritized for model development.

### F. Feature Importance

Feature importance quantifies how much each variable contributes to a model's predictions. Highly important features have a substantial influence on survival prediction, whereas variables with low importance contribute very little and can be removed. Feature importance also improves model interpretability by identifying which biological and clinical variables drive predictions.

# 1. Imports and Processing

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import VarianceThreshold 

In [ ]:
X_train = pd.read_csv("../data/processed/train_features.csv")
X_test = pd.read_csv("../data/processed/test_features.csv")

y_time_train = pd.read_csv("../data/processed/train_time.csv").squeeze()

y_event_train = pd.read_csv("../data/processed/train_event.csv").squeeze()

#print(X_train.shape) verify

# 2. Variance Threshold 

In [3]:
selector = VarianceThreshold(
    threshold=0.01
)

X_train_var = selector.fit_transform(
    X_train
)

X_test_var = selector.transform(
    X_test
)

In [ ]:
#remove low variance features from the dataset and keep only the selected features
selected_features = X_train.columns[
    selector.get_support()
]

X_train_var = pd.DataFrame(
    X_train_var,
    columns=selected_features
)

X_test_var = pd.DataFrame(
    X_test_var,
    columns=selected_features
)

print(X_train_var.shape)

In [ ]:
# sanity check to see if the correct (~identical) features were removed after variance thresholding
removed = X_train.columns[
    ~selector.get_support()
]

print(f"Removed features: {len(removed)}")

removed.tolist()[:30]